In [ ]:
실습 1 — churn_logit1.csv
문제

구독형 서비스 기업이 고객 이탈 여부(churn)를 예측하기 위해age, usage_hour, complaint_cnt를 설명변수로 사용하였다.어떤 변수가 이탈에 영향을 주는지 로지스틱 회귀로 분석하라.

강의 포인트
• 종속변수: churn(0/1)
• 설명변수: age, usage_hour, complaint_cnt
• 확인할 것:
1. 회귀계수 부호
2. p-value 유의성
3. 오즈비
4. 해석 문장

Python 실습 코드
# [실습 1] churn_logit1.csv 로지스틱 회귀: 이탈 여부와 오즈비 해석

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from pathlib import Path

DATA_DIR = Path('/content/type3')  # 필요시 수정
df = pd.read_csv(DATA_DIR / 'churn_logit1.csv')

print("=== 데이터 미리보기 ===")
print(df.head())
print("\n=== 기본 정보 ===")
print(df.info())

# 로지스틱 회귀 적합
model = smf.logit('churn ~ age + usage_hour + complaint_cnt', data=df).fit()

print("\n=== 로지스틱 회귀 요약 ===")
print(model.summary())

# 계수표 정리
result_table = pd.DataFrame({
    'coef': model.params,
    'p_value': model.pvalues,
    'odds_ratio': np.exp(model.params)
})

print("\n=== 계수 / p-value / 오즈비 ===")
print(result_table)

# 95% 신뢰구간과 오즈비 기준으로도 보기
conf = model.conf_int() #interval
conf.columns = ['2.5%', '97.5%']
conf['OR_2.5%'] = np.exp(conf['2.5%'])   #odd비는 exp
conf['OR_97.5%'] = np.exp(conf['97.5%'])

print("\n=== 계수 신뢰구간 및 오즈비 신뢰구간 ===")
print(conf)

# 예측확률 및 분류
df['pred_prob'] = model.predict(df)
df['pred_class'] = (df['pred_prob'] >= 0.5).astype(int) #결과를 정수로

print("\n=== 예측확률 상위 5개 ===")
print(df[['churn', 'pred_prob', 'pred_class']].head())

# 해석용 출력
print("\n=== 해석 가이드 ===")
for var in ['age', 'usage_hour', 'complaint_cnt']:
    coef = model.params[var]
    pval = model.pvalues[var]
    or_val = np.exp(coef)
    direction = '증가' if coef > 0 else '감소'
    print(f"{var}: 계수={coef:.4f}, p-value={pval:.6f}, 오즈비={or_val:.4f}")
    print(f" -> {var}가 1단위 증가할 때 이탈 odds는 약 {or_val:.4f}배가 되며, 방향은 {direction}입니다.")

강사용 해석 포인트

제공된 샘플 결과 기준으로는 다음과 같이 정리할 수 있습니다.

변수
예시 계수
p-value
오즈비
해석
age
-0.0234
0.1064
0.9769
유의하지 않음
usage_hour
-0.0384
0.0000215
0.9623
월 이용시간이 많을수록 이탈 odds 감소
complaint_cnt
0.6561
0.000000259
1.9272
불만건수 1건 증가 시 이탈 odds 약 1.93배


모범 해석(p-value 와 오즈비에 대한 해석)
“로지스틱 회귀분석 결과, usage_hour와 complaint_cnt는 고객 이탈 여부에 유의한 영향을 미쳤고, age는 유의하지 않았다. 특히 complaint_cnt의 회귀계수는 0.6561이며 오즈비는 1.9272로 나타나, 최근 불만건수가 1건 증가할 때 고객 이탈 odds가 약 1.93배 증가한다고 해석할 수 있다. 반면 usage_hour의 계수는 음수이므로 월 이용시간이 많을수록 이탈 가능성은 낮아지는 경향이 있다.”

확인 질문
실습 1용 질문

질문
확인 포인트
complaint_cnt의 계수가 양수라는 것은 무엇을 의미하는가?
불만건수가 늘수록 이탈 가능성 증가
usage_hour의 오즈비가 1보다 작으면 어떻게 해석하는가?
이용시간이 많을수록 이탈 odds 감소
age의 p-value가 0.05보다 크면 무엇을 의미하는가?
유의한 변수라고 보기 어려움




8) 실습 2 — loan_default_logit2.csv
문제
은행은 고객의 대출 연체 여부(default)를 예측하기 위해income, debt_ratio, late_cnt를 설명변수로 사용하였다.이 모형의 적합도와 예측 성능을 평가하라.

강의 포인트
• 종속변수: default(0/1)
• 설명변수: income, debt_ratio, late_cnt
• 확인할 것:
1. 회귀계수와 오즈비
2. 로그우도(llf)
3. residual deviance
4. accuracy, error rate

Python 실습 코드
# [실습 2] loan_default_logit2.csv 로지스틱 회귀: 분류 성능과 residual deviance

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.metrics import accuracy_score
from pathlib import Path

DATA_DIR = Path('/content/type3')  # 필요시 수정
df = pd.read_csv(DATA_DIR / 'loan_default_logit2.csv')

print("=== 데이터 미리보기 ===")
print(df.head())
print("\n=== 기본 정보 ===")
print(df.info())

# 로지스틱 회귀 적합
model = smf.logit('default ~ income + debt_ratio + late_cnt', data=df).fit()

print("\n=== 로지스틱 회귀 요약 ===")
print(model.summary())

# 계수 / p-value / 오즈비
result_table = pd.DataFrame({
    'coef': model.params,
    'p_value': model.pvalues,
    'odds_ratio': np.exp(model.params)   #오즈비는 exp
})
print("\n=== 계수 / p-value / 오즈비 ===")
print(result_table)

# 로그우도와 residual deviance
llf = model.llf
residual_deviance = -2 * llf

print("\n=== 적합도 지표 ===")
print(f"log-likelihood (llf): {llf:.4f}")
print(f"residual deviance: {residual_deviance:.4f}")

# 예측확률, 예측분류
df['pred_prob'] = model.predict(df)
df['pred_class'] = (df['pred_prob'] >= 0.5).astype(int) #정수로 변환

# 성능지표
acc = accuracy_score(df['default'], df['pred_class'])
err = 1 - acc

print("\n=== 분류 성능 ===")
print(f"accuracy     : {acc:.4f}")
print(f"error rate   : {err:.4f}")

# 예측확률 일부 확인
print("\n=== 예측확률 상위 10개 ===")
print(df[['default', 'pred_prob', 'pred_class']].head(10))

# 신뢰구간
conf = model.conf_int()
conf.columns = ['2.5%', '97.5%']
conf['OR_2.5%'] = np.exp(conf['2.5%'])
conf['OR_97.5%'] = np.exp(conf['97.5%'])

print("\n=== 계수 신뢰구간 및 오즈비 신뢰구간 ===")
print(conf)

# 해석용 출력
print("\n=== 해석 가이드 ===")
for var in ['income', 'debt_ratio', 'late_cnt']:
    coef = model.params[var]
    pval = model.pvalues[var]
    or_val = np.exp(coef)
    print(f"{var}: 계수={coef:.6f}, p-value={pval:.6f}, 오즈비={or_val:.6f}")
